# Перегляд готових вибірок

Інтерактивне доповнення до `scripts/checks.py` (той пише `reports/checks.md`
і графіки автоматично). Тут зручно подивитися дані руками перед навчанням.

Формат: `X [N, window_samples, 6]` float32 у фізичних одиницях (acc у g,
gyro у °/с), вікно 2.0 с з перекриттям 50 %. Дані **не нормалізовані**.
Фактичні частота і розмір вікна друкуються нижче — вони читаються з файлів.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
PROC = ROOT / cfg["paths"]["processed"]

splits = {}
for name in ("train", "val", "test"):
    path = PROC / f"{name}.npz"
    if path.exists():
        splits[name] = dict(np.load(path, allow_pickle=False))
    else:
        print(f"немає {path} — запусти python scripts/run_all.py")

train = splits["train"]
CLASSES = [str(c) for c in train["class_names"]]
RAW_CLASSES = [str(c) for c in train["raw_class_names"]]
CHANNELS = [str(c) for c in train["channels"]]
UNITS = {"acc": str(train["units"][0]), "gyr": str(train["units"][1])}
FS = float(train["fs"])
stats = json.loads((PROC / "norm_stats.json").read_text(encoding="utf-8"))

print(f"{FS:g} Гц, вікно {int(train['window_samples'])} × {len(CHANNELS)}, "
      f"крок {int(train['stride_samples'])}")
print(f"канали: {CHANNELS}, одиниці: {UNITS}")
print(f"класи: {dict(enumerate(CLASSES))}")

## Розміри вибірок, учасники, баланс класів

In [ ]:
rows = []
subject_sets = {}
for name, d in splits.items():
    subject_sets[name] = set(d["subject"].tolist())
    rows.append({"split": name, "вікон": d["X"].shape[0], "форма X": str(d["X"].shape),
                 "тип": str(d["X"].dtype), "учасники": sorted(subject_sets[name]),
                 "NaN/inf": int((~np.isfinite(d["X"])).sum())})
display(pd.DataFrame(rows).set_index("split"))

leaks = [(a, b, sorted(subject_sets[a] & subject_sets[b]))
         for a in splits for b in splits if a < b and subject_sets[a] & subject_sets[b]]
print("ВИТІК:", leaks) if leaks else print("OK: учасники не перетинаються")

counts = pd.DataFrame({name: np.bincount(d["y"], minlength=len(CLASSES))
                       for name, d in splits.items()}, index=CLASSES)
display(counts)
ax = (counts / counts.sum() * 100).plot.bar(figsize=(9, 4), width=0.8)
ax.set_ylabel("частка вікон, %")
ax.set_title("Баланс класів (дисбаланс не вирівнюємо — ваги класів при навчанні)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Одиниці: |acc| у спокої має бути ≈ 1 g

In [ ]:
mag = {}
for name, d in splits.items():
    for i, cls in enumerate(CLASSES):
        sel = d["X"][d["y"] == i]
        if sel.shape[0]:
            mag[(name, cls)] = float(np.linalg.norm(sel[:, :, :3], axis=2).mean())
table = pd.Series(mag).unstack().round(3)
display(table)
print("Lying і Stationary мають бути ≈ 1.0 g; Running помітно більше — це нормально.")

ranges = pd.DataFrame({ch: [float(train["X"][:, :, i].min()), float(train["X"][:, :, i].max())]
                       for i, ch in enumerate(CHANNELS)}, index=["min", "max"]).round(2)
display(ranges)

## Приклади вікон по класах (6 каналів)

In [ ]:
rng = np.random.default_rng(int(cfg["seed"]))
t = np.arange(train["X"].shape[1]) / FS

fig, axes = plt.subplots(len(CLASSES), 2, figsize=(12, 2.3 * len(CLASSES)), sharex=True)
for i, cls in enumerate(CLASSES):
    idx = np.flatnonzero(train["y"] == i)
    if idx.size == 0:
        continue
    w = train["X"][rng.choice(idx)]
    for c in range(3):
        axes[i][0].plot(t, w[:, c], lw=1, label=CHANNELS[c])
        axes[i][1].plot(t, w[:, 3 + c], lw=1, label=CHANNELS[3 + c])
    axes[i][0].set_ylabel(f"{cls}\nacc, {UNITS['acc']}", fontsize=8)
    axes[i][1].set_ylabel(f"gyro, {UNITS['gyr']}", fontsize=8)
    for ax in axes[i]:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7, ncol=3, loc="upper right")
axes[-1][0].set_xlabel("час, с")
axes[-1][1].set_xlabel("час, с")
plt.tight_layout()
plt.show()

In [ ]:
# Середній спектр |acc| по класах: ходьба ≈ 1.5-2.5 Гц, біг ≈ 2.5-3.5 Гц.
acc_mag = np.linalg.norm(train["X"][:, :, :3], axis=2)
freqs = np.fft.rfftfreq(acc_mag.shape[1], d=1 / FS)
plt.figure(figsize=(9, 4.5))
for i, cls in enumerate(CLASSES):
    sel = acc_mag[train["y"] == i]
    if sel.shape[0] == 0:
        continue
    spec = np.abs(np.fft.rfft(sel - sel.mean(axis=1, keepdims=True), axis=1)).mean(axis=0)
    plt.semilogy(freqs, spec, lw=1.2, label=cls)
plt.xlabel("частота, Гц")
plt.ylabel("середня амплітуда")
plt.title("Середній спектр |acc| за класами")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Як подавати дані в модель

In [ ]:
mean = np.array(stats["mean"], dtype="float32")
std = np.array(stats["std"], dtype="float32")
Xn = (train["X"] - mean) / std                  # нормалізація статистиками train
print(f"після нормалізації: mean={Xn.mean():.4f}, std={Xn.std():.4f}")
print(f"тільки акселерометр: {train['X'][..., :3].shape}")

n = np.bincount(train["y"], minlength=len(CLASSES))
weights = n.sum() / (len(CLASSES) * np.maximum(n, 1))
display(pd.Series(weights.round(3), index=CLASSES, name="вага класу для навчання"))